# 01 — Data Collection: Polymarket SPX Up or Down

Fetches historical P(Up) probability time series from Polymarket for all
`spx-up-or-down-on-{date}` daily binary contracts.

**Pipeline:**
1. Discover all SPX close-direction markets via Gamma API (`tag_id=102683`)
2. Fetch per-token price history from CLOB API (`/prices-history`)
3. Save raw JSON per contract to `data/polymarket/`
4. Parse into long-format DataFrame and write to `data_processed/polymarket_probabilities.parquet`

**APIs used (public, no auth required):**
- Gamma: `https://gamma-api.polymarket.com` — market discovery & metadata
- CLOB: `https://clob.polymarket.com` — intraday price history

**Known limitation:** Resolved older markets often return empty price history.
Data typically extends back only ~2–4 weeks from collection date.

In [1]:
import re
import json
import time
import logging
from pathlib import Path
from datetime import date

import requests
import pandas as pd
import pyarrow  # noqa: F401 — ensure pyarrow is available for parquet
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pytz
from tqdm.notebook import tqdm

# ── API base URLs ────────────────────────────────────────────────────────────
GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE  = "https://clob.polymarket.com"

SPX_TAG_ID = 102683  # "spx" tag on Gamma API

# Strict regex: close-to-close contracts only
# Excludes: spx-opens-up-or-down-*, spx-above-*, what-will-spx-hit-*, etc.
SLUG_RE = re.compile(r"^spx-up-or-down-on-([a-z]+)-(\d{1,2})-(\d{4})$")

# ── Data directories ─────────────────────────────────────────────────────────
DATA_RAW_DIR  = Path("data/polymarket")
DATA_PROC_DIR = Path("data_processed")
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

# ── HTTP session ─────────────────────────────────────────────────────────────
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json"})

print("Setup complete.")

Setup complete.


In [2]:
def _get(url: str, params: dict | None = None,
         retries: int = 3, backoff: float = 1.5) -> dict | list:
    """GET with exponential backoff. Returns parsed JSON or raises RuntimeError."""
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.get(url, params=params, timeout=15)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as exc:
            last_exc = exc
            wait = backoff ** attempt
            log.warning(
                "Attempt %d/%d failed for %s — retrying in %.1fs: %s",
                attempt, retries, url, wait, exc,
            )
            time.sleep(wait)
    raise RuntimeError(f"All {retries} retries failed for {url}") from last_exc

## 1. Market Discovery

Paginate through `GET /events?tag_id=102683` to collect all SPX close-direction contracts.

**Token ordering (empirically confirmed):**
- `clobTokenIds[0]` = "Up" outcome token
- `clobTokenIds[1]` = "Down" outcome token

**Slug filter:** Only `spx-up-or-down-on-{month}-{day}-{year}` — excludes
`spx-opens-up-or-down-*` (open-to-open contracts) and other variants.

Expected: ~115 contracts from ~October 2025 to present.

In [ ]:
def discover_spx_markets(
    tag_id: int = SPX_TAG_ID,
    page_size: int = 100,
) -> pd.DataFrame:
    """
    Discover all SPX close-direction markets via Gamma API.

    Returns DataFrame with columns:
        slug, condition_id, up_token_id, down_token_id,
        start_date, end_date, volume, closed
    """
    records = []
    offset = 0

    while True:
        data = _get(
            f"{GAMMA_BASE}/events",
            params={"tag_id": tag_id, "limit": page_size, "offset": offset},
        )

        if not data:
            break  # exhausted

        for event in data:
            for mkt in event.get("markets", []):
                slug = mkt.get("slug", "")
                if not SLUG_RE.match(slug):
                    continue  # skip opens-*, above-*, etc.

                token_ids = mkt.get("clobTokenIds", [])
                if len(token_ids) != 2:
                    log.warning(
                        "Unexpected token count for %s: %s", slug, token_ids
                    )
                    continue

                # volume field name varies across API versions
                volume_raw = mkt.get("volumeNum") or mkt.get("volume") or 0

                records.append({
                    "slug":          slug,
                    "condition_id":  mkt.get("conditionId", ""),
                    "up_token_id":   token_ids[0],
                    "down_token_id": token_ids[1],
                    "start_date":    pd.Timestamp(mkt.get("startDate"), tz="UTC"),
                    "end_date":      pd.Timestamp(mkt.get("endDate"),   tz="UTC"),
                    "volume":        float(volume_raw),
                    "closed":        bool(mkt.get("closed", False)),
                })

        if len(data) < page_size:
            break  # last (partial) page
        offset += page_size
    df = (
        pd.DataFrame(records)
        .drop_duplicates(subset="slug")
        .sort_values("end_date")
        .reset_index(drop=True)
    )
    log.info("Discovered %d SPX close-direction markets", len(df))
    return df


markets_df = discover_spx_markets()
print(f"Markets found: {len(markets_df)}")
if len(markets_df):
    print(f"Date range : {markets_df['end_date'].min().date()} → {markets_df['end_date'].max().date()}")
    print(f"Closed     : {markets_df['closed'].sum()} / {len(markets_df)}")
markets_df.head(10)

2026-04-08 17:46:46,556 [WARNING] Unexpected token count for spx-up-or-down-on-october-15-2025: ["66108394744999606650312348788205408992947540593856466994916687636887951265038", "48749066617222889147309885754120162201231181017188075681227535616431063552669"]
2026-04-08 17:46:46,556 [WARNING] Unexpected token count for spx-up-or-down-on-october-16-2025: ["92235036752649464522665560183518761541701731215613499717631975701905874864408", "108210271641774348895248892659198448017571269927018507833398413156192612918913"]
2026-04-08 17:46:46,557 [WARNING] Unexpected token count for spx-up-or-down-on-october-17-2025: ["64208447568099210767095013250647079260678318575781913827872509087525941223976", "50946884718434269894406291288446755117542087655327683214382113407845045328871"]
2026-04-08 17:46:46,557 [WARNING] Unexpected token count for spx-up-or-down-on-october-20-2025: ["87655548706954159971526433734261563414507569947327284542510958929098830795480", "3981392452758913805748943587733879890978954

test []


KeyError: 'end_date'

## 2. Price History Fetching

For each market's `up_token_id`, call:
```
GET https://clob.polymarket.com/prices-history?market={token_id}&interval=max
```

Response: `{"history": [{"t": unix_seconds, "p": probability}, ...]}`

**Idempotent:** If `data/polymarket/{slug}.json` already exists the HTTP call is skipped.
Re-run with `force_refresh=True` to re-fetch everything.

**Rate limiting:** 0.15 s sleep between calls (~6 req/s, well under the 9 000 req/10 s limit).

In [ ]:
def fetch_price_history(
    slug: str,
    token_id: str,
    force_refresh: bool = False,
) -> list[dict] | None:
    """
    Fetch CLOB price history for the Up outcome token.
    Saves raw JSON to data/polymarket/{slug}.json.

    Returns list of {t, p} dicts or None if no data is available.
    """
    out_path = DATA_RAW_DIR / f"{slug}.json"

    if out_path.exists() and not force_refresh:
        with open(out_path) as fh:
            payload = json.load(fh)
        history = payload.get("history") or []
        return history if history else None

    try:
        data = _get(
            f"{CLOB_BASE}/prices-history",
            params={"market": token_id, "interval": "max"},
        )
    except RuntimeError as exc:
        log.error("Failed to fetch history for %s: %s", slug, exc)
        return None

    # Always save raw payload — even if empty, so future runs skip the call
    with open(out_path, "w") as fh:
        json.dump(data, fh)

    history = data.get("history", [])
    if not history:
        log.warning(
            "Empty history for %s (resolved market — data likely purged)", slug
        )
        return None

    return history


def fetch_all_histories(
    markets: pd.DataFrame,
    force_refresh: bool = False,
) -> dict[str, list]:
    """
    Fetch price histories for all markets.
    Returns {slug: history_list} for markets that have data.
    """
    results: dict[str, list] = {}
    failed:  list[str]       = []

    for _, row in tqdm(markets.iterrows(), total=len(markets),
                       desc="Fetching price histories"):
        history = fetch_price_history(
            row["slug"], row["up_token_id"], force_refresh=force_refresh
        )
        if history is not None:
            results[row["slug"]] = history
        else:
            failed.append(row["slug"])
        time.sleep(0.15)  # polite rate limit

    log.info(
        "Fetched %d / %d markets (%d empty/failed)",
        len(results), len(markets), len(failed),
    )
    if failed:
        log.warning("No history for %d markets (first few): %s",
                    len(failed), failed[:5])
    return results


histories = fetch_all_histories(markets_df, force_refresh=False)
print(f"Markets with data : {len(histories)} / {len(markets_df)}")

## 3. Data Processing

Parse slug → `contract_date` and raw `{t, p}` lists → long-format DataFrame.

In [ ]:
MONTH_MAP: dict[str, int] = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}


def parse_contract_date(slug: str) -> date | None:
    """Extract contract date from slug like 'spx-up-or-down-on-march-3-2026'."""
    m = SLUG_RE.match(slug)
    if not m:
        return None
    month = MONTH_MAP.get(m.group(1))
    if month is None:
        return None
    try:
        return date(int(m.group(3)), month, int(m.group(2)))
    except ValueError:
        return None


# Sanity checks
assert parse_contract_date("spx-up-or-down-on-march-3-2026")   == date(2026, 3, 3)
assert parse_contract_date("spx-up-or-down-on-october-15-2025") == date(2025, 10, 15)
assert parse_contract_date("spx-up-or-down-on-april-9-2026")   == date(2026, 4, 9)
assert parse_contract_date("spx-opens-up-or-down-on-march-3-2026") is None
print("parse_contract_date: OK")

In [ ]:
def build_long_dataframe(
    histories: dict[str, list],
) -> pd.DataFrame:
    """
    Parse all raw history dicts into a single long-format DataFrame.

    Output columns:
        slug          : str
        contract_date : datetime64[ns]       (date contract resolves)
        timestamp_utc : datetime64[ns, UTC]  (tick timestamp)
        p_up          : float64              (P(Up) ∈ [0, 1])
    """
    frames = []

    for slug, history in histories.items():
        contract_date = parse_contract_date(slug)
        if contract_date is None:
            log.warning("Could not parse date from slug: %s — skipping", slug)
            continue

        df = pd.DataFrame(history, columns=["t", "p"])
        df = df.rename(columns={"t": "timestamp_utc", "p": "p_up"})

        # CLOB timestamps are Unix seconds (not milliseconds)
        df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], unit="s", utc=True)
        df["p_up"]          = df["p_up"].astype("float64")
        df["slug"]          = slug
        df["contract_date"] = pd.Timestamp(contract_date)

        # Drop exact duplicate timestamps (API occasionally emits repeated ticks)
        df = df.drop_duplicates(subset="timestamp_utc").sort_values("timestamp_utc")

        frames.append(df[["slug", "contract_date", "timestamp_utc", "p_up"]])

    if not frames:
        raise ValueError(
            "No data frames to concatenate — all histories were empty. "
            "Resolved markets may have had their price history purged."
        )

    result = pd.concat(frames, ignore_index=True)

    log.info(
        "Long DataFrame: %d rows, %d contracts, %s → %s",
        len(result),
        result["slug"].nunique(),
        result["contract_date"].min().date(),
        result["contract_date"].max().date(),
    )
    return result


prob_df = build_long_dataframe(histories)
print(prob_df.dtypes)
print(f"\nShape: {prob_df.shape}")
prob_df.head()

## 4. Save to Parquet

In [ ]:
OUT_PATH = DATA_PROC_DIR / "polymarket_probabilities.parquet"

prob_df.to_parquet(OUT_PATH, index=False, engine="pyarrow")

# Verify round-trip
check = pd.read_parquet(OUT_PATH)
assert len(check) == len(prob_df), "Row count mismatch after parquet round-trip!"

print(f"Saved {len(prob_df):,} rows → {OUT_PATH}")
print(f"File size: {OUT_PATH.stat().st_size / 1024:.1f} KB")

## 5. Validation

Check: `p_up ∈ [0, 1]`, reasonable tick counts per contract, no large gaps in business-day coverage.

In [ ]:
# ── p_up range ────────────────────────────────────────────────────────────────
assert prob_df["p_up"].between(0, 1).all(), "p_up has out-of-range values!"
print(f"p_up range: [{prob_df['p_up'].min():.4f}, {prob_df['p_up'].max():.4f}]")

# ── Ticks per contract ────────────────────────────────────────────────────────
ticks_per = (
    prob_df.groupby("slug")["timestamp_utc"]
    .count()
    .rename("tick_count")
    .sort_values()
)
print("\nTicks per contract (distribution):")
print(ticks_per.describe().round(1))
print(f"\nContracts with < 5 ticks: {(ticks_per < 5).sum()}")

# ── Coverage ──────────────────────────────────────────────────────────────────
print(f"\nContracts in dataset : {prob_df['slug'].nunique()}")
print(f"Date range           : {prob_df['contract_date'].min().date()} → {prob_df['contract_date'].max().date()}")

# ── Missing business days (rough check) ───────────────────────────────────────
all_dates   = pd.to_datetime(prob_df["contract_date"].unique()).sort_values()
biz_days    = pd.bdate_range(all_dates.min(), all_dates.max())
covered     = {d.date() for d in all_dates}
missing_biz = [d for d in biz_days if d.date() not in covered]
if missing_biz:
    print(f"\nBusiness days with no contract data ({len(missing_biz)}):")
    for d in missing_biz[:20]:
        print(f"  {d.date()}")
else:
    print("\nNo missing business days detected.")

## 6. Sample P(Up) Trajectories

Plot the 4 most data-rich contracts to visually verify the probability paths look reasonable
(should start near 0.5 and converge toward 0 or 1 as contract resolution approaches).

In [ ]:
ET = pytz.timezone("America/New_York")

# Pick the 4 contracts with the most ticks
sample_slugs = (
    prob_df.groupby("slug")["timestamp_utc"]
    .count()
    .nlargest(4)
    .index.tolist()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
axes = axes.flatten()

for ax, slug in zip(axes, sample_slugs):
    sub = prob_df[prob_df["slug"] == slug].copy()
    sub["ts_et"] = sub["timestamp_utc"].dt.tz_convert(ET)

    ax.plot(sub["ts_et"], sub["p_up"], linewidth=1.5, color="#0072B2")
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel("P(Up)")
    ax.set_title(slug.replace("spx-up-or-down-on-", ""), fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M", tz=ET))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right", fontsize=7)

fig.suptitle("P(Up) intraday trajectories — sample contracts (ET)", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_PROC_DIR / "sample_p_up_trajectories.png", dpi=120)
plt.show()

## 7. Summary Table

In [ ]:
summary = (
    prob_df.groupby("slug")
    .agg(
        contract_date=("contract_date", "first"),
        ticks=("p_up", "count"),
        p_up_open=("p_up", "first"),
        p_up_close=("p_up", "last"),
        p_up_mean=("p_up", "mean"),
        p_up_std=("p_up", "std"),
    )
    .reset_index()
    .sort_values("contract_date")
)

# Merge resolution status from markets_df
closed_map = markets_df.set_index("slug")["closed"].to_dict()
summary["resolved"] = summary["slug"].map(closed_map)

print(f"Summary table: {len(summary)} contracts")
summary.tail(10)